In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, r2_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
    ConfusionMatrixDisplay, classification_report
)
import warnings
warnings.filterwarnings('ignore')
print("libraries imported!")

In [ ]:
df = pd.read_csv('../dataset/heart_statlog_cleveland_hungary_final.csv')
print(df.shape)
df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
X = df.drop('target', axis=1)
y = df['target']
print(X.shape, y.shape)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("scaling done!")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f"train size: {X_train.shape}, test size: {X_test.shape}")

In [ ]:
svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svr.fit(X_train, y_train)
print("model trained!")

## Regression Metrics (Original)

In [ ]:
y_pred = svr.predict(X_test)
y_pred_binary = (y_pred >= 0.5).astype(int)

mse = mean_squared_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f"MSE      : {mse:.4f}")
print(f"R2 Score : {r2:.4f}")
print(f"Accuracy : {(y_pred_binary == y_test).mean()*100:.2f}%")

## Classification Metrics (Precision, Recall, F1-Score, AUC-ROC)

SVR outputs continuous scores.  
- **Precision, Recall, F1-Score** are computed from binarised predictions (threshold 0.5).  
- **AUC-ROC** uses the raw continuous SVR scores directly — no thresholding needed.

In [ ]:
precision = precision_score(y_test, y_pred_binary, zero_division=0)
recall    = recall_score(y_test, y_pred_binary)
f1        = f1_score(y_test, y_pred_binary)
auc_roc   = roc_auc_score(y_test, y_pred)   # raw SVR scores → best AUC

print("="*50)
print("  SVR — Classification Performance Summary")
print("  (heart_statlog_cleveland_hungary_final)")
print("="*50)
print(f"  Accuracy  : {(y_pred_binary == y_test).mean()*100:.2f}%")
print(f"  Precision : {precision:.4f}")
print(f"  Recall*   : {recall:.4f}   ← clinical priority")
print(f"  F1-Score  : {f1:.4f}")
print(f"  AUC-ROC   : {auc_roc:.4f}")
print(f"  MSE       : {mse:.4f}")
print(f"  R² Score  : {r2:.4f}")
print("="*50)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_binary,
                             target_names=['No Disease (0)', 'Heart Disease (1)']))

## ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#E8724C', lw=2,
         label=f'SVR (AUC = {auc_roc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.10, color='#E8724C')
plt.xlim([0, 1])
plt.ylim([0, 1.02])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate',  fontsize=12)
plt.title('SVR — ROC Curve\nHeart Disease Classification', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig('svr_roc_curve.png', bbox_inches='tight')
plt.show()
print('[✓] Saved → svr_roc_curve.png')

## Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_binary),
    display_labels=['No Disease', 'Disease']
).plot(ax=ax, colorbar=False, cmap='Oranges')
ax.set_title('SVR — Confusion Matrix', fontsize=12, fontweight='bold', color='#E8724C', pad=10)
ax.grid(False)
plt.tight_layout()
plt.savefig('svr_confusion_matrix.png', bbox_inches='tight')
plt.show()
print('[✓] Saved → svr_confusion_matrix.png')

## Actual vs Predicted (Original Plot)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(y_test.values[:50], label='actual', marker='o')
plt.plot(y_pred[:50], label='predicted', marker='x')
plt.title('SVR - Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.savefig('svr_performance.png')
plt.show()
print("plot saved!")